# 05.03 — XGBoost Baseline

## Purpose

Train the first real binary classifier using only the 17 Phase 4 predictors. The
validation period controls early stopping; the test period remains untouched.

## Setup and reproducibility

In [1]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CURRENT_PATH = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (CURRENT_PATH, *CURRENT_PATH.parents)
     if (path / "README.md").is_file() and (path / ".git").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the movie_rating_predictor repository root")

FEATURE_PATH = PROJECT_ROOT / "data" / "features" / "rating_features_v1.parquet"
FEATURE_METADATA_PATH = PROJECT_ROOT / "data" / "features" / "rating_features_v1.metadata.json"
RATINGS_PATH = PROJECT_ROOT / "data" / "processed" / "ratings.parquet"
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_COLUMNS = [
    "global_rating_count",
    "global_mean_rating",
    "user_rating_count",
    "user_mean_rating",
    "user_rating_std_pop",
    "user_seconds_since_last_rating",
    "movie_rating_count",
    "movie_mean_rating",
    "movie_rating_std_pop",
    "movie_rating_count_30d",
    "user_target_genre_rating_count",
    "user_target_genre_mean_rating",
    "user_target_genre_mean_delta",
    "movie_genre_count",
    "movie_release_year",
    "movie_release_year_missing",
    "movie_age_years",
]

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Random seed:", RANDOM_SEED)
print("Feature count:", len(FEATURE_COLUMNS))
print("Features:", FEATURE_COLUMNS)


TRAIN_END = pd.Timestamp("2012-01-01")
VALIDATION_END = pd.Timestamp("2014-01-01")

SPLIT_DEFINITIONS = {
    "train": {"start": None, "end": TRAIN_END},
    "validation": {"start": TRAIN_END, "end": VALIDATION_END},
    "test": {"start": VALIDATION_END, "end": None},
}

def split_mask(timestamp: pd.Series, split_name: str) -> np.ndarray:
    bounds = SPLIT_DEFINITIONS[split_name]
    mask = np.ones(len(timestamp), dtype=bool)
    if bounds["start"] is not None:
        mask &= timestamp.to_numpy() >= bounds["start"].to_datetime64()
    if bounds["end"] is not None:
        mask &= timestamp.to_numpy() < bounds["end"].to_datetime64()
    return mask

Python: 3.14.6
pandas: 3.0.5
NumPy: 2.5.3
Random seed: 42
Feature count: 17
Features: ['global_rating_count', 'global_mean_rating', 'user_rating_count', 'user_mean_rating', 'user_rating_std_pop', 'user_seconds_since_last_rating', 'movie_rating_count', 'movie_mean_rating', 'movie_rating_std_pop', 'movie_rating_count_30d', 'user_target_genre_rating_count', 'user_target_genre_mean_rating', 'user_target_genre_mean_delta', 'movie_genre_count', 'movie_release_year', 'movie_release_year_missing', 'movie_age_years']


In [2]:
import os
import time
import sklearn
import xgboost as xgb
from xgboost import XGBClassifier

print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("Logical CPUs:", os.cpu_count())
print("XGBoost build info:", xgb.build_info())

scikit-learn: 1.9.1
XGBoost: 3.4.1
Logical CPUs: 10
XGBoost build info: {'BUILTIN_PREFETCH_PRESENT': True, 'CLANG_VERSION': [15, 0, 0], 'DEBUG': False, 'MM_PREFETCH_PRESENT': False, 'USE_CUDA': False, 'USE_DLOPEN_NCCL': False, 'USE_FEDERATED': False, 'USE_NCCL': False, 'USE_OPENMP': True, 'USE_RMM': False, 'libxgboost': '/Users/angel/Documents/movie_rating_predictor/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib'}


## Load canonical outcomes and temporal partitions

Parquet predicate pushdown reads only train and validation rows. Predictor values are
converted once to compact `float32`; intentional missing values remain `NaN` for
XGBoost's native missing routing.

In [3]:
rating_values = pd.read_parquet(RATINGS_PATH, columns=["rating"])["rating"].to_numpy(dtype=np.float32, copy=False)

def load_split(split_name: str, extra_columns=()):
    """Load one half-open temporal partition and attach labels by event identity."""
    bounds = SPLIT_DEFINITIONS[split_name]
    filters = []
    if bounds["start"] is not None:
        filters.append(("timestamp", ">=", bounds["start"]))
    if bounds["end"] is not None:
        filters.append(("timestamp", "<", bounds["end"]))

    columns = list(dict.fromkeys(["ratingEventId", "timestamp", *FEATURE_COLUMNS, *extra_columns]))
    frame = pd.read_parquet(FEATURE_PATH, columns=columns, filters=filters or None)
    if bounds["start"] is not None:
        frame = frame.loc[frame["timestamp"] >= bounds["start"]]
    if bounds["end"] is not None:
        frame = frame.loc[frame["timestamp"] < bounds["end"]]

    event_ids = frame["ratingEventId"].to_numpy(dtype=np.uint64, copy=False)
    if event_ids.min() < 1 or event_ids.max() > len(rating_values):
        raise AssertionError("ratingEventId is outside the canonical ratings row range")
    y = (rating_values[event_ids - 1] >= 4.0).astype(np.int8)
    return frame.reset_index(drop=True), y

def as_float32_matrix(frame: pd.DataFrame) -> np.ndarray:
    """Create the single compact dense matrix handed to XGBoost."""
    return frame.loc[:, FEATURE_COLUMNS].to_numpy(dtype=np.float32, copy=True)

In [4]:
train_frame, y_train = load_split("train")
validation_frame, y_validation = load_split("validation")

X_train = as_float32_matrix(train_frame)
X_validation = as_float32_matrix(validation_frame)
train_range = (train_frame["timestamp"].min(), train_frame["timestamp"].max())
validation_range = (validation_frame["timestamp"].min(), validation_frame["timestamp"].max())
del train_frame, validation_frame

print("Train:", X_train.shape, "prevalence:", y_train.mean(), "range:", train_range)
print("Validation:", X_validation.shape, "prevalence:", y_validation.mean(), "range:", validation_range)
assert X_train.shape[1] == X_validation.shape[1] == 17

Train: (17822773, 17) prevalence: 0.498194136232336 range: (Timestamp('1995-01-09 11:46:44'), Timestamp('2011-12-31 23:59:55'))
Validation: (1330716, 17) prevalence: 0.5195391052636326 range: (Timestamp('2012-01-01 00:00:40'), Timestamp('2013-12-31 23:59:59'))


## Conservative baseline configuration

Histogram trees, 8 CPU threads, 128 bins, row/column subsampling, regularization, and
early stopping make the full 20M-scale baseline practical on a CPU machine. This is
one baseline configuration, not a hyperparameter search.

In [5]:
MODEL_PARAMS = {
    "objective": "binary:logistic",
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "min_child_weight": 20,
    "reg_alpha": 0.1,
    "reg_lambda": 2.0,
    "max_bin": 128,
    "tree_method": "hist",
    "eval_metric": ["logloss", "auc", "aucpr"],
    "early_stopping_rounds": 30,
    "random_state": RANDOM_SEED,
    "n_jobs": min(8, os.cpu_count() or 1),
    "missing": np.nan,
}

model = XGBClassifier(**MODEL_PARAMS)
print(model.get_params())

{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.9, 'device': None, 'early_stopping_rounds': 30, 'enable_categorical': True, 'eval_metric': ['logloss', 'auc', 'aucpr'], 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.05, 'max_bin': 128, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': 20, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 600, 'n_jobs': 8, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': 0.1, 'reg_lambda': 2.0, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}


## Fit with validation early stopping

In [6]:
fit_started = time.perf_counter()
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_validation, y_validation)],
    verbose=25,
)
training_runtime_seconds = time.perf_counter() - fit_started
best_iteration = int(model.best_iteration)
print(f"Training runtime: {training_runtime_seconds:,.1f} seconds")
print("Best iteration (zero-based):", best_iteration)
print("Effective trees:", best_iteration + 1)

[0]	validation_0-logloss:0.68167	validation_0-auc:0.77925	validation_0-aucpr:0.76228	validation_1-logloss:0.68178	validation_1-auc:0.77666	validation_1-aucpr:0.77367


[25]	validation_0-logloss:0.57376	validation_0-auc:0.78730	validation_0-aucpr:0.77086	validation_1-logloss:0.57253	validation_1-auc:0.78522	validation_1-aucpr:0.78331


[50]	validation_0-logloss:0.55538	validation_0-auc:0.78992	validation_0-aucpr:0.77350	validation_1-logloss:0.55596	validation_1-auc:0.78757	validation_1-aucpr:0.78576


[75]	validation_0-logloss:0.55065	validation_0-auc:0.79162	validation_0-aucpr:0.77534	validation_1-logloss:0.55172	validation_1-auc:0.78966	validation_1-aucpr:0.78828


[100]	validation_0-logloss:0.54863	validation_0-auc:0.79275	validation_0-aucpr:0.77652	validation_1-logloss:0.54983	validation_1-auc:0.79102	validation_1-aucpr:0.78976


...

[525]	validation_0-logloss:0.54204	validation_0-auc:0.79794	validation_0-aucpr:0.78279	validation_1-logloss:0.54301	validation_1-auc:0.79707	validation_1-aucpr:0.79699


[550]	validation_0-logloss:0.54187	validation_0-auc:0.79809	validation_0-aucpr:0.78297	validation_1-logloss:0.54290	validation_1-auc:0.79717	validation_1-aucpr:0.79710


[575]	validation_0-logloss:0.54172	validation_0-auc:0.79822	validation_0-aucpr:0.78313	validation_1-logloss:0.54282	validation_1-auc:0.79726	validation_1-aucpr:0.79720


[599]	validation_0-logloss:0.54160	validation_0-auc:0.79832	validation_0-aucpr:0.78326	validation_1-logloss:0.54278	validation_1-auc:0.79731	validation_1-aucpr:0.79728


Training runtime: 2,649.1 seconds
Best iteration (zero-based): 597
Effective trees: 598


## Train and validation metrics

Accuracy is intentionally omitted. PR-AUC is interpreted alongside prevalence.

In [7]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

def binary_metrics(y_true, probability):
    return {
        "Rows": len(y_true),
        "Prevalence": float(np.mean(y_true)),
        "ROC-AUC": roc_auc_score(y_true, probability),
        "PR-AUC": average_precision_score(y_true, probability),
        "Log Loss": log_loss(y_true, probability, labels=[0, 1]),
        "Brier": brier_score_loss(y_true, probability),
    }

In [8]:
train_probability = model.predict_proba(X_train)[:, 1]
validation_probability = model.predict_proba(X_validation)[:, 1]
metric_table = pd.DataFrame({
    "train": binary_metrics(y_train, train_probability),
    "validation": binary_metrics(y_validation, validation_probability),
}).T
display(metric_table)
del train_probability, validation_probability

,Rows,Prevalence,ROC-AUC,PR-AUC,Log Loss,Brier
train,17822773.0,0.498194,0.798314,0.783246,0.541610,0.182881
validation,1330716.0,0.519539,0.797312,0.797281,0.542776,0.183097


## Persist the model and run metadata

In [9]:
MODELS_DIR.mkdir(exist_ok=True)
model_path = MODELS_DIR / "xgboost_baseline_v1.json"
model.save_model(model_path)

run_metadata = {
    "random_seed": RANDOM_SEED,
    "feature_columns": FEATURE_COLUMNS,
    "model_params": {key: (value.tolist() if isinstance(value, np.ndarray) else value) for key, value in MODEL_PARAMS.items() if key != "missing"},
    "missing": "NaN (native XGBoost handling)",
    "training_runtime_seconds": training_runtime_seconds,
    "best_iteration_zero_based": best_iteration,
    "effective_tree_count": best_iteration + 1,
    "train_timestamp_range": [value.isoformat() for value in train_range],
    "validation_timestamp_range": [value.isoformat() for value in validation_range],
    "metrics": metric_table.to_dict(orient="index"),
    "package_versions": {"xgboost": xgb.__version__, "scikit_learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
metadata_path = MODELS_DIR / "xgboost_baseline_v1.metadata.json"
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
print("Saved model:", model_path.relative_to(PROJECT_ROOT))
print("Saved metadata:", metadata_path.relative_to(PROJECT_ROOT))

Saved model: models/xgboost_baseline_v1.json
Saved metadata: models/xgboost_baseline_v1.metadata.json
